In [1]:
%cd ..

/Users/v1adych/inno/data-mining/group_project


In [2]:
import polars as pl

In [3]:
events = pl.scan_parquet("data/events.parquet")

pr = (
    events
    .filter(pl.col("event_type") == "PullRequestEvent")
    .select(
        "event_id",
        "created_at",
        "actor_id",
        "actor_login",
        "repo_id",
        "repo_name",
        "payload_json",
    )
    .with_columns(
        pl.col("created_at")
          .str.strptime(pl.Datetime, "%Y-%m-%dT%H:%M:%SZ")
          .alias("created_at_dt"),

        pl.col("payload_json").str.json_path_match("$.action").alias("action"),
        pl.col("payload_json").str.json_path_match("$.number").cast(pl.Int64).alias("pr_number"),
        pl.col("payload_json").str.json_path_match("$.pull_request.id").cast(pl.Int64).alias("pr_id"),

        pl.col("payload_json").str.json_path_match("$.pull_request.merged").alias("merged_raw"),
        pl.col("payload_json").str.json_path_match("$.pull_request.author_association").alias("author_association"),
        pl.col("payload_json").str.json_path_match("$.pull_request.draft").alias("draft_raw"),

        pl.col("payload_json").str.json_path_match("$.pull_request.commits").cast(pl.Int64).alias("commits"),
        pl.col("payload_json").str.json_path_match("$.pull_request.additions").cast(pl.Int64).alias("additions"),
        pl.col("payload_json").str.json_path_match("$.pull_request.deletions").cast(pl.Int64).alias("deletions"),
        pl.col("payload_json").str.json_path_match("$.pull_request.changed_files").cast(pl.Int64).alias("changed_files"),

        pl.col("payload_json").str.json_path_match("$.pull_request.comments").cast(pl.Int64).alias("comments"),
        pl.col("payload_json").str.json_path_match("$.pull_request.review_comments").cast(pl.Int64).alias("review_comments"),
    )
    .collect()
)

pr.shape

(470260, 20)

In [4]:
opened = (
    pr
    .filter(pl.col("action") == "opened")
    .sort("created_at_dt")
    .group_by("pr_id")
    .agg(
        pl.col("event_id").first().alias("opened_event_id"),
        pl.col("created_at_dt").first().alias("opened_at"),
        pl.col("actor_id").first().alias("opening_actor_id"),
        pl.col("actor_login").first().alias("opening_actor_login"),
        pl.col("repo_id").first().alias("repo_id"),
        pl.col("repo_name").first().alias("repo_name"),
        pl.col("pr_number").first().alias("pr_number"),
        pl.col("author_association").first().alias("author_association"),
        pl.col("draft_raw").first().alias("draft_raw"),
        pl.col("commits").first().alias("commits"),
        pl.col("additions").first().alias("additions"),
        pl.col("deletions").first().alias("deletions"),
        pl.col("changed_files").first().alias("changed_files"),
        pl.col("comments").first().alias("comments_at_open"),
        pl.col("review_comments").first().alias("review_comments_at_open"),
    )
)

closed = (
    pr
    .filter(pl.col("action") == "closed")
    .sort("created_at_dt")
    .group_by("pr_id")
    .agg(
        pl.col("event_id").first().alias("closed_event_id"),
        pl.col("created_at_dt").first().alias("closed_at"),
        pl.col("merged_raw").first().alias("merged_raw"),
    )
)

lifecycle = (
    opened
    .join(closed, on="pr_id", how="inner")
    .filter(pl.col("closed_at") > pl.col("opened_at"))
    .with_columns(
        (pl.col("merged_raw") == "true").cast(pl.Int8).alias("merged"),
        ((pl.col("closed_at") - pl.col("opened_at")).dt.total_hours()).alias("time_to_close_hours"),
        pl.col("opened_at").dt.hour().alias("opened_hour"),
        pl.col("opened_at").dt.weekday().alias("opened_weekday"),
        (pl.col("opened_at").dt.weekday() >= 6).cast(pl.Int8).alias("opened_on_weekend"),
        (pl.col("draft_raw") == "true").cast(pl.Int8).alias("draft"),
    )
)

lifecycle.shape

(195425, 25)

In [5]:
lifecycle.select(
    pl.len().alias("rows"),
    pl.col("merged").mean().alias("merge_rate"),
    pl.col("time_to_close_hours").median().alias("median_time_to_close_hours"),
    pl.col("time_to_close_hours").quantile(0.9).alias("p90_time_to_close_hours"),
    pl.col("commits").null_count().alias("missing_commits"),
    pl.col("additions").null_count().alias("missing_additions"),
    pl.col("changed_files").null_count().alias("missing_changed_files"),
)

rows,merge_rate,median_time_to_close_hours,p90_time_to_close_hours,missing_commits,missing_additions,missing_changed_files
u32,f64,f64,f64,u32,u32,u32
195425,0.857383,0.0,45.0,0,0,0


In [6]:
event_history = (
    events
    .select(
        "created_at",
        "event_type",
        "actor_id",
        "repo_id",
    )
    .with_columns(
        pl.col("created_at")
          .str.strptime(pl.Datetime, "%Y-%m-%dT%H:%M:%SZ")
          .alias("event_time")
    )
    .select("event_time", "event_type", "actor_id", "repo_id")
    .collect()
)

event_history.shape

(9269533, 4)

In [7]:
event_daily = (
    event_history
    .with_columns(pl.col("event_time").dt.date().alias("event_date"))
    .group_by("event_date", "repo_id")
    .agg(
        pl.len().alias("repo_events_that_day"),
        pl.col("actor_id").n_unique().alias("repo_unique_actors_that_day"),
        pl.col("event_type").n_unique().alias("repo_event_type_diversity_that_day"),
        (pl.col("event_type") == "PullRequestEvent").sum().alias("repo_pr_events_that_day"),
        (pl.col("event_type") == "IssueCommentEvent").sum().alias("repo_issue_comments_that_day"),
        (pl.col("event_type") == "PushEvent").sum().alias("repo_push_events_that_day"),
    )
    .sort("repo_id", "event_date")
    .with_columns(
        pl.col("repo_events_that_day").cum_sum().over("repo_id").alias("repo_events_cum_including_day"),
        pl.col("repo_unique_actors_that_day").cum_sum().over("repo_id").alias("repo_unique_actor_days_cum_including_day"),
        pl.col("repo_pr_events_that_day").cum_sum().over("repo_id").alias("repo_pr_events_cum_including_day"),
        pl.col("repo_issue_comments_that_day").cum_sum().over("repo_id").alias("repo_issue_comments_cum_including_day"),
        pl.col("repo_push_events_that_day").cum_sum().over("repo_id").alias("repo_push_events_cum_including_day"),
    )
    .with_columns(
        (
            pl.col("repo_events_cum_including_day") - pl.col("repo_events_that_day")
        ).alias("repo_events_before_day"),

        (
            pl.col("repo_pr_events_cum_including_day") - pl.col("repo_pr_events_that_day")
        ).alias("repo_pr_events_before_day"),

        (
            pl.col("repo_issue_comments_cum_including_day") - pl.col("repo_issue_comments_that_day")
        ).alias("repo_issue_comments_before_day"),

        (
            pl.col("repo_push_events_cum_including_day") - pl.col("repo_push_events_that_day")
        ).alias("repo_push_events_before_day"),

        (
            pl.col("repo_unique_actor_days_cum_including_day") - pl.col("repo_unique_actors_that_day")
        ).alias("repo_actor_days_before_day"),
    )
    .select(
        "event_date",
        "repo_id",
        "repo_events_before_day",
        "repo_pr_events_before_day",
        "repo_issue_comments_before_day",
        "repo_push_events_before_day",
        "repo_actor_days_before_day",
    )
)

In [8]:
actor_daily = (
    event_history
    .with_columns(pl.col("event_time").dt.date().alias("event_date"))
    .group_by("event_date", "actor_id")
    .agg(
        pl.len().alias("actor_events_that_day"),
        pl.col("repo_id").n_unique().alias("actor_unique_repos_that_day"),
        (pl.col("event_type") == "PullRequestEvent").sum().alias("actor_pr_events_that_day"),
        (pl.col("event_type") == "IssueCommentEvent").sum().alias("actor_issue_comments_that_day"),
        (pl.col("event_type") == "PushEvent").sum().alias("actor_push_events_that_day"),
    )
    .sort("actor_id", "event_date")
    .with_columns(
        pl.col("actor_events_that_day").cum_sum().over("actor_id").alias("actor_events_cum_including_day"),
        pl.col("actor_unique_repos_that_day").cum_sum().over("actor_id").alias("actor_repo_days_cum_including_day"),
        pl.col("actor_pr_events_that_day").cum_sum().over("actor_id").alias("actor_pr_events_cum_including_day"),
        pl.col("actor_issue_comments_that_day").cum_sum().over("actor_id").alias("actor_issue_comments_cum_including_day"),
        pl.col("actor_push_events_that_day").cum_sum().over("actor_id").alias("actor_push_events_cum_including_day"),
    )
    .with_columns(
        (
            pl.col("actor_events_cum_including_day") - pl.col("actor_events_that_day")
        ).alias("actor_events_before_day"),

        (
            pl.col("actor_repo_days_cum_including_day") - pl.col("actor_unique_repos_that_day")
        ).alias("actor_repo_days_before_day"),

        (
            pl.col("actor_pr_events_cum_including_day") - pl.col("actor_pr_events_that_day")
        ).alias("actor_pr_events_before_day"),

        (
            pl.col("actor_issue_comments_cum_including_day") - pl.col("actor_issue_comments_that_day")
        ).alias("actor_issue_comments_before_day"),

        (
            pl.col("actor_push_events_cum_including_day") - pl.col("actor_push_events_that_day")
        ).alias("actor_push_events_before_day"),
    )
    .select(
        "event_date",
        "actor_id",
        "actor_events_before_day",
        "actor_repo_days_before_day",
        "actor_pr_events_before_day",
        "actor_issue_comments_before_day",
        "actor_push_events_before_day",
    )
)

In [9]:
dataset = (
    lifecycle
    .with_columns(pl.col("opened_at").dt.date().alias("opened_date"))
    .join(
        event_daily,
        left_on=["opened_date", "repo_id"],
        right_on=["event_date", "repo_id"],
        how="left",
    )
    .join(
        actor_daily,
        left_on=["opened_date", "opening_actor_id"],
        right_on=["event_date", "actor_id"],
        how="left",
    )
    .with_columns(
        pl.col("author_association").fill_null("UNKNOWN"),
        pl.col("draft").fill_null(0),

        pl.col("commits").fill_null(0),
        pl.col("additions").fill_null(0),
        pl.col("deletions").fill_null(0),
        pl.col("changed_files").fill_null(0),
        pl.col("comments_at_open").fill_null(0),
        pl.col("review_comments_at_open").fill_null(0),

        pl.col("repo_events_before_day").fill_null(0),
        pl.col("repo_pr_events_before_day").fill_null(0),
        pl.col("repo_issue_comments_before_day").fill_null(0),
        pl.col("repo_push_events_before_day").fill_null(0),
        pl.col("repo_actor_days_before_day").fill_null(0),

        pl.col("actor_events_before_day").fill_null(0),
        pl.col("actor_repo_days_before_day").fill_null(0),
        pl.col("actor_pr_events_before_day").fill_null(0),
        pl.col("actor_issue_comments_before_day").fill_null(0),
        pl.col("actor_push_events_before_day").fill_null(0),
    )
)

dataset.shape

(195425, 36)

In [10]:
dataset.select(
    pl.len().alias("rows"),
    pl.col("merged").mean().alias("merge_rate"),
    pl.col("repo_id").n_unique().alias("unique_repos"),
    pl.col("opening_actor_id").n_unique().alias("unique_opening_actors"),
    pl.col("time_to_close_hours").median().alias("median_time_to_close_hours"),
    pl.col("time_to_close_hours").quantile(0.9).alias("p90_time_to_close_hours"),
)

rows,merge_rate,unique_repos,unique_opening_actors,median_time_to_close_hours,p90_time_to_close_hours
u32,f64,u32,u32,f64,f64
195425,0.857383,59624,62011,0.0,45.0


In [11]:
dataset.write_parquet("data/pr_dataset.parquet", compression="zstd")

In [12]:
dataset.schema

Schema([('pr_id', Int64),
        ('opened_event_id', String),
        ('opened_at', Datetime(time_unit='us', time_zone=None)),
        ('opening_actor_id', Int64),
        ('opening_actor_login', String),
        ('repo_id', Int64),
        ('repo_name', String),
        ('pr_number', Int64),
        ('author_association', String),
        ('draft_raw', String),
        ('commits', Int64),
        ('additions', Int64),
        ('deletions', Int64),
        ('changed_files', Int64),
        ('comments_at_open', Int64),
        ('review_comments_at_open', Int64),
        ('closed_event_id', String),
        ('closed_at', Datetime(time_unit='us', time_zone=None)),
        ('merged_raw', String),
        ('merged', Int8),
        ('time_to_close_hours', Int64),
        ('opened_hour', Int8),
        ('opened_weekday', Int8),
        ('opened_on_weekend', Int8),
        ('draft', Int8),
        ('opened_date', Date),
        ('repo_events_before_day', UInt32),
        ('repo_pr_events_befo